# QTI_MLP synthetic/patient training modes

Four supervised `QTI_MLP` modes are available without editing any `.py` files:
`patient_only`, `synthetic_only`, `mixed`, and `pretrain_then_patient`.

This notebook reuses the existing thesis recreation notebook for the patient
fold logic and reuses selected helper cells from
`C:\QTI_ML\QTI_MLP_practice_synth_train_predict_p1_p14_patho.ipynb`
for the synthetic pathology DTD stack handling.


In [ ]:
from pathlib import Path
from datetime import datetime
import json, sys, time, random, hashlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\qti_mlp_baseline")
SOURCE_SYNTH_NOTEBOOK = Path(r"C:\QTI_ML\QTI_MLP_practice_synth_train_predict_p1_p14_patho.ipynb")
DATA_ROOT = Path(r"B:\ML_TS")
SYN_ROOT = Path(r"C:\SynQTI-IR")
SYNTHETIC_ROOT = SYN_ROOT / "data" / "DTDs_cov_suite_2_patho"
SYNTHETIC_XPS = SYN_ROOT / "data" / "xps" / "xps_sub_min_pp.mat"
SYNTHETIC_STACK_ROOT = SYN_ROOT / "data" / "Results_SNR20_50_MLP_patho_stacks_scalar_R1500"

OUTPUT_ROOT = PROJECT_ROOT / "synthetic_patient_training_outputs"
MODEL_ROOT = OUTPUT_ROOT / "models"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
TENSORBOARD_ROOT = PROJECT_ROOT / "runs" / "synthetic_patient_training_modes"

TRAIN_MODE = "pretrain_then_patient"  # patient_only | synthetic_only | mixed | pretrain_then_patient
SMOKE_MODE = True
OVERWRITE_OUTPUTS = False
SEED = 42

PATIENT_ORDER = ["P1","P10","P2","P11","P3","P12","P4","P13","P5","P14","P6","P15","P7","P16","P8","P17","P9","P18"]
NII_NAME = "og_NII_sub_min_dn_db_dg_tp_mc_b0_avg.nii.gz"
MASK_NAME = "manual_mask.nii.gz"
LABEL_REL = Path("qtip_sub_min_pipe_2") / "dps3_nan_clmpd_clmpd.mat"
INVAR_KEYS = ["MD", "FA", "uFA", "C_c", "C_MD"]
SLICE_IND = [[2, 21] for _ in PATIENT_ORDER]

TS_SIZE, VS_SIZE = 1, 1
ENS_FACTOR = len(PATIENT_ORDER)
FOLDS_TO_RUN = [0] if SMOKE_MODE else list(range(ENS_FACTOR))

PATIENT_LR = 1e-3
PATIENT_BATCH_SIZE = 512
PATIENT_EPOCHS = 100
PATIENT_RUN_EPOCHS = 1 if SMOKE_MODE else PATIENT_EPOCHS
PATIENT_DECAY_CONST = 0.925
PATIENT_SCHED_START, PATIENT_SCHED_END = 14, 74
PATIENT_EARLY_STOPPING = True
PATIENT_EARLY_STOP_MIN_EPOCHS = 15
PATIENT_EARLY_STOP_PATIENCE = 8
PATIENT_EARLY_STOP_MIN_DELTA = 1e-6

SYNTHETIC_SNR_LEVELS = [20, 25, 30, 35, 40, 45, 50]
SYNTHETIC_N_REALIZATIONS = 1500
SYNTHETIC_EPOCHS = 5
SYNTHETIC_RUN_EPOCHS = 1 if SMOKE_MODE else SYNTHETIC_EPOCHS
SYNTHETIC_LR = 1e-3
SYNTHETIC_BATCH_SIZE = None  # None -> 4096 on cuda, else 1024
SYNTHETIC_VALIDATION_FRACTION = 0.10
SYNTHETIC_VALIDATION_STRIDE = int(round(1 / SYNTHETIC_VALIDATION_FRACTION))
SYNTHETIC_VALIDATION_OFFSET = 0
SYNTHETIC_OVERWRITE_STACKS = False
SYNTHETIC_RUN_GENERATION_SMOKE_TEST = True
SYNTHETIC_MAX_CASES = 5 if SMOKE_MODE else None
ALLOWED_SCENARIOS = None

MIXED_OUTPUT_SCALE = "synthetic"
MIXED_PATIENT_STEPS_PER_EPOCH = None
MIXED_SYNTHETIC_STEPS_PER_EPOCH = None

FINAL_SIGMOID = False
LAYER_NORM = False
BIAS = True
DEVICE = "cpu"  # change to "cuda" manually if your env supports it
NUM_WORKERS = 0
PIN_MEMORY = DEVICE == "cuda"

for d in [OUTPUT_ROOT, MODEL_ROOT, PREDICTION_ROOT, METRICS_ROOT, MANIFEST_ROOT, FIGURE_ROOT, TENSORBOARD_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

assert TRAIN_MODE in {"patient_only", "synthetic_only", "mixed", "pretrain_then_patient"}
print("mode:", TRAIN_MODE, "| smoke:", SMOKE_MODE, "| folds:", FOLDS_TO_RUN)


In [ ]:
import importlib.util
required = ["torch", "nibabel", "numpy", "pandas", "scipy", "matplotlib", "tensorboard"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Missing dependencies: " + ", ".join(missing) + ". Activate QTI_Project or install requirements.txt.")

for root in [PROJECT_ROOT, SYN_ROOT]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))


In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.io as sio
import torch
from torch import nn
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from IPython.display import display

from QTI_MLP import QTI_Dataset, QTI_MLP, train_loop, test_loop, rev_zscore
from qti_phantom import QTI_Phantom
from utils.dtd_math import c_m, c_md, c_mu, compute_cumulant_tensors, fa, md, ufa, v_md
from utils.dtd_utils import calc_dtens
print("torch:", torch.__version__)


In [ ]:
# Reuse synthetic helper definitions from the existing synthetic-pathology notebook.
# Cells 3 and 4 define SyntheticCase, GT utilities, stack generation, manifest creation,
# SyntheticSNRStackDataset, and the synthetic weighted target z-score helpers.
if not SOURCE_SYNTH_NOTEBOOK.exists():
    raise FileNotFoundError(SOURCE_SYNTH_NOTEBOOK)
synth_nb = json.loads(SOURCE_SYNTH_NOTEBOOK.read_text(encoding="utf-8"))
patho_btens_file = SYNTHETIC_XPS
for cell_idx in [3, 4]:
    exec("".join(synth_nb["cells"][cell_idx]["source"]), globals())
print("Loaded synthetic helper cells from:", SOURCE_SYNTH_NOTEBOOK)


In [ ]:
def set_deterministic_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def build_patient_table():
    rows = []
    for patient in PATIENT_ORDER:
        base = DATA_ROOT / patient
        rows.append({"patient": patient, "nii": base / NII_NAME, "mask": base / MASK_NAME, "label": base / LABEL_REL})
    return pd.DataFrame(rows)

patient_table = build_patient_table()
checks = []
for _, row in patient_table.iterrows():
    checks.append({
        "patient": row["patient"],
        "nii_exists": row["nii"].exists(),
        "mask_exists": row["mask"].exists(),
        "label_exists": row["label"].exists(),
        "nii": str(row["nii"]),
        "mask": str(row["mask"]),
        "label": str(row["label"]),
    })
checks = pd.DataFrame(checks)
display(checks)
bad = checks.loc[~(checks["nii_exists"] & checks["mask_exists"] & checks["label_exists"])]
if len(bad):
    raise FileNotFoundError("Missing patient inputs: " + ", ".join(bad["patient"].tolist()))
if not SYNTHETIC_ROOT.exists():
    raise FileNotFoundError(SYNTHETIC_ROOT)
if not SYNTHETIC_XPS.exists():
    raise FileNotFoundError(SYNTHETIC_XPS)
print("Validated patient and synthetic roots.")


In [ ]:
def make_fold(current_step):
    current_ind = np.arange(len(PATIENT_ORDER))
    current_ind = np.roll(current_ind, shift=-current_step * TS_SIZE)
    ts_ind = current_ind[:TS_SIZE].tolist()
    vs_ind = current_ind[TS_SIZE:TS_SIZE + VS_SIZE].tolist()
    tr_ind = current_ind[TS_SIZE + VS_SIZE:].tolist()
    return {
        "fold": current_step,
        "test_idx": ts_ind,
        "val_idx": vs_ind,
        "train_idx": tr_ind,
        "test_patients": [PATIENT_ORDER[i] for i in ts_ind],
        "val_patients": [PATIENT_ORDER[i] for i in vs_ind],
        "train_patients": [PATIENT_ORDER[i] for i in tr_ind],
    }

fold_table = pd.DataFrame([
    {"fold": make_fold(i)["fold"], "test": ",".join(make_fold(i)["test_patients"]), "validation": ",".join(make_fold(i)["val_patients"]), "n_train": len(make_fold(i)["train_patients"])}
    for i in range(ENS_FACTOR)
])
display(fold_table)
assert all(fold_table["n_train"] == 16)

def paths_for(indices, column):
    return [str(patient_table.iloc[i][column]) for i in indices]

def slices_for(indices):
    return [SLICE_IND[i] for i in indices]

def prepare_patient_dataset(indices, target_scale="patient", y_mu=None, y_sd=None):
    ds = QTI_Dataset(paths_for(indices, "nii"), paths_for(indices, "label"), paths_for(indices, "mask"), slices_for(indices), INVAR_KEYS, zscore_output=(target_scale == "patient"))
    ds.thresh_neg_vals(); ds.mask_zero_vals(); ds.mask_nan_vals()
    ds.apply_masked_tensor(); ds.zscore_norm_input()
    if target_scale == "patient":
        ds.zscore_norm_labels(INVAR_KEYS)
    elif target_scale == "synthetic":
        y = ds.y.to_tensor(float("nan"))
        mu = torch.tensor(np.asarray(y_mu, dtype=np.float32).reshape(1, 1, 1, 1, -1))
        sd = torch.tensor(np.asarray(y_sd, dtype=np.float32).reshape(1, 1, 1, 1, -1))
        ds.y = (y - mu) / sd
        ds.zscore = torch.tensor(np.vstack([y_mu, y_sd]), dtype=torch.float32)
    else:
        raise ValueError(target_scale)
    ds.flatten_slice_dim(); ds.apply_mask()
    return ds

def make_loader(ds, batch_size, shuffle, generator=None):
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, pin_memory=PIN_MEMORY, num_workers=NUM_WORKERS, generator=generator)


In [ ]:
def run_epoch_weighted(loader, model, loss_fn, optimizer):
    model.train()
    total_loss = 0.0; total_n = 0
    for X, y in loader:
        X = X.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(model(X), y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * X.shape[0]; total_n += X.shape[0]
    return total_loss / max(total_n, 1)

def evaluate_epoch_weighted(loader, model, loss_fn):
    model.eval()
    total_loss = 0.0; total_n = 0
    with torch.no_grad():
        for X, y in loader:
            X = X.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
            loss = loss_fn(model(X), y)
            total_loss += loss.item() * X.shape[0]; total_n += X.shape[0]
    return total_loss / max(total_n, 1)

def train_patient_phase(model, fold, run_id, epochs, target_scale="patient", y_mu=None, y_sd=None):
    ds_train = prepare_patient_dataset(fold["train_idx"], target_scale, y_mu, y_sd)
    ds_val = prepare_patient_dataset(fold["val_idx"], target_scale, y_mu, y_sd)
    train_loader = make_loader(ds_train, PATIENT_BATCH_SIZE, True)
    val_loader = make_loader(ds_val, PATIENT_BATCH_SIZE, True)
    optimizer = torch.optim.Adam(model.parameters(), lr=PATIENT_LR)
    scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=PATIENT_DECAY_CONST)
    loss_fn = nn.MSELoss()
    writer = SummaryWriter(str(TENSORBOARD_ROOT / run_id / f"fold_{fold['fold']}_patient"))
    hist = []; best = float("inf"); best_epoch = 0; wait = 0; stopped = False
    for epoch in range(epochs):
        tr = train_loop(train_loader, model, loss_fn, optimizer, DEVICE, PATIENT_BATCH_SIZE)
        va = test_loop(val_loader, model, loss_fn, DEVICE, PATIENT_BATCH_SIZE)
        if PATIENT_SCHED_START <= epoch <= PATIENT_SCHED_END:
            scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        if va < best - PATIENT_EARLY_STOP_MIN_DELTA:
            best = va; best_epoch = epoch + 1; wait = 0
        else:
            wait += 1
        hist.append({"phase": "patient", "epoch": epoch + 1, "train_loss": tr, "val_loss": va, "lr": lr, "best_val_loss": best, "wait": wait})
        writer.add_scalars("Training vs. Validation Loss", {"Training": tr, "Validation": va}, epoch + 1)
        writer.add_scalars("Learning Rate", {"LR": lr}, epoch + 1); writer.flush()
        print(f"patient fold {fold['fold']} epoch {epoch+1:03d}/{epochs}: train={tr:.6g} val={va:.6g} best={best:.6g} wait={wait}/{PATIENT_EARLY_STOP_PATIENCE} lr={lr:.3g}")
        if PATIENT_EARLY_STOPPING and epoch + 1 >= PATIENT_EARLY_STOP_MIN_EPOCHS and wait >= PATIENT_EARLY_STOP_PATIENCE:
            stopped = True
            break
    writer.close()
    zscore = ds_train.zscore.detach().cpu().numpy() if hasattr(ds_train.zscore, "detach") else np.asarray(ds_train.zscore)
    return model, hist, zscore, {"epochs": len(hist), "best_epoch": best_epoch, "best_val_loss": best, "stopped_early": stopped}


In [ ]:
def prepare_synthetic_training():
    set_deterministic_seed(SEED)
    cases = collect_synthetic_cases(SYNTHETIC_ROOT, allowed_scenarios=ALLOWED_SCENARIOS)
    if SYNTHETIC_MAX_CASES is not None:
        cases = cases[:SYNTHETIC_MAX_CASES]
    print("synthetic candidate cases:", len(cases))
    if not cases:
        raise RuntimeError("No synthetic cases found.")
    if SYNTHETIC_RUN_GENERATION_SMOKE_TEST:
        run_generation_smoke_test(cases[0], SYNTHETIC_SNR_LEVELS[0], SYNTHETIC_STACK_ROOT, stable_seed(cases[0].scenario_name, cases[0].case_name, "smoke", base_seed=SEED))
    manifest, n_scans = prepare_noisy_snr_manifest(cases, INVAR_KEYS, SYNTHETIC_SNR_LEVELS, SYNTHETIC_N_REALIZATIONS, SYNTHETIC_STACK_ROOT, SEED, overwrite=SYNTHETIC_OVERWRITE_STACKS)
    y_mu, y_sd = weighted_target_zscore(manifest, INVAR_KEYS)
    train_ds = SyntheticSNRStackDataset(manifest, INVAR_KEYS, y_mu, y_sd, "train", SYNTHETIC_VALIDATION_STRIDE, SYNTHETIC_VALIDATION_OFFSET)
    val_ds = SyntheticSNRStackDataset(manifest, INVAR_KEYS, y_mu, y_sd, "val", SYNTHETIC_VALIDATION_STRIDE, SYNTHETIC_VALIDATION_OFFSET)
    batch = SYNTHETIC_BATCH_SIZE or (4096 if DEVICE == "cuda" else 1024)
    gen = torch.Generator(device="cpu").manual_seed(SEED)
    train_loader = make_loader(train_ds, batch, True, gen)
    val_loader = make_loader(val_ds, batch, False)
    snr_presence = make_snr_presence_table(manifest, SYNTHETIC_SNR_LEVELS, SYNTHETIC_VALIDATION_STRIDE)
    display(snr_presence)
    print(f"synthetic stacks={len(manifest)} train rows={len(train_ds):,} val rows={len(val_ds):,} n_scans={n_scans}")
    return {"manifest": manifest, "n_scans": n_scans, "y_mu": y_mu, "y_sd": y_sd, "train_loader": train_loader, "val_loader": val_loader, "snr_presence": snr_presence}

def train_synthetic_phase(model, synth, run_id, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=SYNTHETIC_LR)
    loss_fn = nn.MSELoss()
    writer = SummaryWriter(str(TENSORBOARD_ROOT / run_id / "synthetic"))
    hist = []
    for epoch in range(epochs):
        tr = run_epoch_weighted(synth["train_loader"], model, loss_fn, optimizer)
        va = evaluate_epoch_weighted(synth["val_loader"], model, loss_fn)
        hist.append({"phase": "synthetic", "epoch": epoch + 1, "train_loss": tr, "val_loss": va, "lr": SYNTHETIC_LR})
        writer.add_scalars("Training vs. Validation Loss", {"Training": tr, "Validation": va}, epoch + 1); writer.flush()
        print(f"synthetic epoch {epoch+1:03d}/{epochs}: train={tr:.6g} val={va:.6g}")
    writer.close()
    return model, hist, np.vstack([synth["y_mu"], synth["y_sd"]])


In [ ]:
def mixed_epoch(patient_loader, synth_loader, model, loss_fn, optimizer):
    model.train()
    p_iter = iter(patient_loader); s_iter = iter(synth_loader)
    p_steps = MIXED_PATIENT_STEPS_PER_EPOCH or len(patient_loader)
    s_steps = MIXED_SYNTHETIC_STEPS_PER_EPOCH or p_steps
    p_loss = s_loss = 0.0; p_n = s_n = 0
    for step in range(max(p_steps, s_steps)):
        if step < p_steps:
            try: X, y = next(p_iter)
            except StopIteration: p_iter = iter(patient_loader); X, y = next(p_iter)
            X = X.to(DEVICE); y = y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True); loss = loss_fn(model(X), y); loss.backward(); optimizer.step()
            p_loss += loss.item() * X.shape[0]; p_n += X.shape[0]
        if step < s_steps:
            try: X, y = next(s_iter)
            except StopIteration: s_iter = iter(synth_loader); X, y = next(s_iter)
            X = X.to(DEVICE); y = y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True); loss = loss_fn(model(X), y); loss.backward(); optimizer.step()
            s_loss += loss.item() * X.shape[0]; s_n += X.shape[0]
    return p_loss / max(p_n, 1), s_loss / max(s_n, 1)

def train_mixed_fold(fold, synth, run_id, epochs):
    ds_train = prepare_patient_dataset(fold["train_idx"], "synthetic", synth["y_mu"], synth["y_sd"])
    ds_val = prepare_patient_dataset(fold["val_idx"], "synthetic", synth["y_mu"], synth["y_sd"])
    patient_loader = make_loader(ds_train, PATIENT_BATCH_SIZE, True)
    patient_val_loader = make_loader(ds_val, PATIENT_BATCH_SIZE, False)
    model = QTI_MLP(synth["n_scans"], len(INVAR_KEYS), LAYER_NORM, FINAL_SIGMOID, BIAS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=PATIENT_LR)
    loss_fn = nn.MSELoss()
    writer = SummaryWriter(str(TENSORBOARD_ROOT / run_id / f"fold_{fold['fold']}_mixed"))
    hist = []
    for epoch in range(epochs):
        p_tr, s_tr = mixed_epoch(patient_loader, synth["train_loader"], model, loss_fn, optimizer)
        p_va = evaluate_epoch_weighted(patient_val_loader, model, loss_fn)
        s_va = evaluate_epoch_weighted(synth["val_loader"], model, loss_fn)
        hist.append({"phase": "mixed", "epoch": epoch + 1, "patient_train_loss": p_tr, "synthetic_train_loss": s_tr, "patient_val_loss": p_va, "synthetic_val_loss": s_va})
        writer.add_scalars("Mixed Loss", {"Patient Train": p_tr, "Synthetic Train": s_tr, "Patient Val": p_va, "Synthetic Val": s_va}, epoch + 1); writer.flush()
        print(f"mixed fold {fold['fold']} epoch {epoch+1:03d}/{epochs}: patient={p_tr:.6g}/{p_va:.6g} synthetic={s_tr:.6g}/{s_va:.6g}")
    writer.close()
    return model, hist, np.vstack([synth["y_mu"], synth["y_sd"]])


In [ ]:
def run_id_now():
    return f"QTI_MLP_SYNTH_PATIENT_{TRAIN_MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

def save_bundle(model, zscore, history, run_id, fold=None, extra=None):
    out = MODEL_ROOT / TRAIN_MODE
    out.mkdir(parents=True, exist_ok=True)
    fold_part = "synthetic" if fold is None else f"fold{fold['fold']:02d}_ts{''.join(fold['test_patients'])}_vs{''.join(fold['val_patients'])}"
    model_path = out / f"{run_id}_{fold_part}.pt"
    if model_path.exists() and not OVERWRITE_OUTPUTS:
        raise FileExistsError(model_path)
    torch.save(model.state_dict(), model_path)
    zscore_path = Path(str(model_path) + "_zscore.csv")
    pd.DataFrame(np.asarray(zscore), index=["mean", "std"], columns=INVAR_KEYS).to_csv(zscore_path, index=False)
    history_path = out / f"{run_id}_{fold_part}_history.csv"
    pd.DataFrame(history).to_csv(history_path, index=False)
    meta = {"run_id": run_id, "mode": TRAIN_MODE, "fold": None if fold is None else fold, "model_path": str(model_path), "zscore_path": str(zscore_path), "history_path": str(history_path)}
    if extra: meta.update(extra)
    settings_path = out / f"{run_id}_{fold_part}_settings.json"
    settings_path.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
    return {"fold": None if fold is None else fold["fold"], "model_path": str(model_path), "zscore_path": str(zscore_path), "history_path": str(history_path), "settings_path": str(settings_path)}

def prepare_eval_dataset(patient):
    idx = PATIENT_ORDER.index(patient)
    ds = QTI_Dataset([str(patient_table.iloc[idx]["nii"])], [str(patient_table.iloc[idx]["label"])], [str(patient_table.iloc[idx]["mask"])], [SLICE_IND[idx]], INVAR_KEYS, zscore_output=False)
    ds.thresh_neg_vals(); ds.mask_zero_vals(); ds.mask_nan_vals()
    ds.apply_masked_tensor(); ds.zscore_norm_input()
    ds.flatten_slice_dim(); ds.apply_mask()
    return ds

def predict_and_evaluate(model, zscore, patients, run_id, tag):
    model.eval()
    zscore_t = torch.tensor(np.asarray(zscore), dtype=torch.float32).to(DEVICE)
    rows = []
    pred_root = PREDICTION_ROOT / TRAIN_MODE / run_id / tag
    pred_root.mkdir(parents=True, exist_ok=True)
    for patient in patients:
        ds = prepare_eval_dataset(patient)
        y_true = ds.y.to_tensor(float("nan")) if hasattr(ds.y, "to_tensor") else ds.y
        y_true = y_true.detach().cpu().numpy()
        with torch.no_grad():
            pred = rev_zscore(model(ds.X.to(DEVICE)), zscore_t).detach().cpu().numpy()
        mask_img = nib.load(str(DATA_ROOT / patient / MASK_NAME))
        mask_xyz = mask_img.get_fdata().astype(bool)
        mask_flat = ds.mask.detach().cpu().numpy().astype(bool).reshape(-1)
        zxy_shape = (mask_xyz.shape[2], mask_xyz.shape[0], mask_xyz.shape[1])
        mask_zxy = mask_flat.reshape(zxy_shape)
        patient_dir = pred_root / patient
        patient_dir.mkdir(parents=True, exist_ok=True)
        for i, key in enumerate(INVAR_KEYS):
            vol_zxy = np.full(zxy_shape, np.nan, dtype=np.float32)
            vol_zxy[mask_zxy] = pred[:, i]
            nib.save(nib.Nifti1Image(np.transpose(vol_zxy, (1, 2, 0)), mask_img.affine), str(patient_dir / f"{run_id}_{tag}_{key}_prediction.nii.gz"))
            valid = np.isfinite(y_true[:, i]) & np.isfinite(pred[:, i])
            err = pred[valid, i] - y_true[valid, i]
            rows.append({"run_id": run_id, "tag": tag, "patient": patient, "invariant": key, "n": int(valid.sum()), "mae": float(np.mean(np.abs(err))), "mse": float(np.mean(err**2)), "rmse": float(np.sqrt(np.mean(err**2))), "bias": float(np.mean(err))})
    metrics = pd.DataFrame(rows)
    metrics_path = METRICS_ROOT / f"{run_id}_{tag}_metrics.csv"
    metrics.to_csv(metrics_path, index=False)
    return metrics, metrics_path


In [ ]:
run_id = run_id_now()
results, metric_frames = [], []
synth = None
if TRAIN_MODE in {"synthetic_only", "mixed", "pretrain_then_patient"}:
    synth = prepare_synthetic_training()
    synth["manifest"].to_csv(MANIFEST_ROOT / f"{run_id}_synthetic_manifest.csv", index=False)
    synth["snr_presence"].to_csv(MANIFEST_ROOT / f"{run_id}_synthetic_snr_presence.csv", index=False)

if TRAIN_MODE == "synthetic_only":
    model = QTI_MLP(synth["n_scans"], len(INVAR_KEYS), LAYER_NORM, FINAL_SIGMOID, BIAS).to(DEVICE)
    model, hist, zscore = train_synthetic_phase(model, synth, run_id, SYNTHETIC_RUN_EPOCHS)
    results.append(save_bundle(model, zscore, hist, run_id, extra={"synthetic_epochs": SYNTHETIC_RUN_EPOCHS}))
    metrics, path = predict_and_evaluate(model, zscore, ["P1"] if SMOKE_MODE else PATIENT_ORDER, run_id, "synthetic_only")
    metric_frames.append(metrics); print("metrics:", path)

elif TRAIN_MODE == "patient_only":
    for fold_id in FOLDS_TO_RUN:
        fold = make_fold(fold_id)
        tmp = prepare_patient_dataset(fold["train_idx"], "patient")
        model = QTI_MLP(tmp.X.size()[-1], len(INVAR_KEYS), LAYER_NORM, FINAL_SIGMOID, BIAS).to(DEVICE)
        del tmp
        model, hist, zscore, info = train_patient_phase(model, fold, run_id, PATIENT_RUN_EPOCHS, "patient")
        results.append(save_bundle(model, zscore, hist, run_id, fold, info))
        metrics, path = predict_and_evaluate(model, zscore, fold["test_patients"], run_id, f"fold{fold_id:02d}_patient")
        metric_frames.append(metrics); print("metrics:", path)

elif TRAIN_MODE == "pretrain_then_patient":
    base = QTI_MLP(synth["n_scans"], len(INVAR_KEYS), LAYER_NORM, FINAL_SIGMOID, BIAS).to(DEVICE)
    base, synth_hist, synth_zscore = train_synthetic_phase(base, synth, run_id, SYNTHETIC_RUN_EPOCHS)
    base_bundle = save_bundle(base, synth_zscore, synth_hist, run_id, extra={"synthetic_epochs": SYNTHETIC_RUN_EPOCHS})
    results.append(base_bundle)
    base_state = {k: v.detach().cpu().clone() for k, v in base.state_dict().items()}
    for fold_id in FOLDS_TO_RUN:
        fold = make_fold(fold_id)
        model = QTI_MLP(synth["n_scans"], len(INVAR_KEYS), LAYER_NORM, FINAL_SIGMOID, BIAS).to(DEVICE)
        model.load_state_dict(base_state)
        model, patient_hist, patient_zscore, info = train_patient_phase(model, fold, run_id, PATIENT_RUN_EPOCHS, "patient")
        hist = [{"phase": "synthetic_pretrain", **h} for h in synth_hist] + [{"phase": "patient_finetune", **h} for h in patient_hist]
        results.append(save_bundle(model, patient_zscore, hist, run_id, fold, {"pretrain_model_path": base_bundle["model_path"], **info}))
        metrics, path = predict_and_evaluate(model, patient_zscore, fold["test_patients"], run_id, f"fold{fold_id:02d}_pretrain_then_patient")
        metric_frames.append(metrics); print("metrics:", path)

elif TRAIN_MODE == "mixed":
    for fold_id in FOLDS_TO_RUN:
        fold = make_fold(fold_id)
        model, hist, zscore = train_mixed_fold(fold, synth, run_id, PATIENT_RUN_EPOCHS)
        results.append(save_bundle(model, zscore, hist, run_id, fold, {"mixed_output_scale": MIXED_OUTPUT_SCALE}))
        metrics, path = predict_and_evaluate(model, zscore, fold["test_patients"], run_id, f"fold{fold_id:02d}_mixed")
        metric_frames.append(metrics); print("metrics:", path)

results_df = pd.DataFrame(results)
results_path = MANIFEST_ROOT / f"{run_id}_model_results.csv"
results_df.to_csv(results_path, index=False)
display(results_df)
if metric_frames:
    all_metrics = pd.concat(metric_frames, ignore_index=True)
    all_metrics_path = METRICS_ROOT / f"{run_id}_all_metrics.csv"
    all_metrics.to_csv(all_metrics_path, index=False)
    display(all_metrics)
    print("all metrics:", all_metrics_path)
print("model manifest:", results_path)


In [ ]:
def plot_histories(results_df):
    for p in results_df["history_path"].dropna().map(Path):
        hist = pd.read_csv(p)
        fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
        if "train_loss" in hist:
            ax.plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
        if "val_loss" in hist:
            ax.plot(hist["epoch"], hist["val_loss"], marker="o", label="val")
        if "patient_train_loss" in hist:
            ax.plot(hist["epoch"], hist["patient_train_loss"], marker="o", label="patient train")
            ax.plot(hist["epoch"], hist["synthetic_train_loss"], marker="o", label="synthetic train")
            ax.plot(hist["epoch"], hist["patient_val_loss"], marker="o", label="patient val")
            ax.plot(hist["epoch"], hist["synthetic_val_loss"], marker="o", label="synthetic val")
        ax.set_title(p.stem); ax.set_xlabel("epoch"); ax.set_ylabel("loss")
        ax.grid(alpha=0.25); ax.legend(frameon=False)
        out = FIGURE_ROOT / f"{p.stem}_loss.png"
        fig.tight_layout(); fig.savefig(out, bbox_inches="tight"); plt.show()
        print("saved", out)

plot_histories(results_df)
